In [1]:
import rapid_code_load_T0 as load
import formation_channels as fc
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colors
from pathlib import Path
import pandas as pd

In [2]:
dat_path = Path("/Users/kbreivik/Data/Synthetic-UCBs/data_products/simulated_binary_populations/monte_carlo_comparisons/")

run_waves = ['initial_condition_variations', 'mass_transfer_variations']
initial_conditions_variations = ['fiducial', 'thermal_ecc', 'uniform_ecc', 
                                 'm2_min_05', 'qmin_01', 'porb_log_uniform']
mass_transfer_variations = ['alpha_lambda_1', 'alpha_lambda_2', 'alpha_lambda_02', 'alpha_lambda_05', 
                            'alpha_gamma_2', 'qcrit_claeys_14','qcrit_hurley_02', 'qcrit_hurley_webbink', 
                            'qcrit_zetas', 'accretion_0', 'accretion_1', 'accretion_05']

codes = ['COSMIC', 'BSE', 'SEVN', 'SeBa', 'COMPAS', 'METISSE', 'ComBinE']
fnames = ['COSMIC_T0.hdf5', 'BSE_T0.hdf5', 'SEVN_MIST_T0.csv', 'SeBa_T0.hdf5', 'COMPAS_T0.hdf5', 'METISSE_T0.hdf5', 'ComBinE_T0.hdf5']

In [3]:
import h5py

outfile = 'channel_ids.h5'

with h5py.File(outfile, 'w') as hf:
    for rw, subtype in zip(run_waves, [initial_conditions_variations, mass_transfer_variations]):
        for s in subtype:
            for c in codes:
                if rw == 'initial_condition_variations':
                    path = dat_path / rw / s
                elif rw == 'mass_transfer_variations':
                    if s in ['alpha_lambda_1', 'alpha_lambda_2', 'alpha_lambda_02', 'alpha_lambda_05', 'alpha_gamma_2']:
                        path = dat_path / rw / 'common_envelope' / s
                    elif s in ['qcrit_claeys_14', 'qcrit_hurley_02', 'qcrit_hurley_webbink', 'qcrit_zetas']:
                        path = dat_path / rw / 'stability_of_mass_transfer' / s
                    elif s in ['accretion_0', 'accretion_1', 'accretion_05']:
                        path = dat_path / rw / 'stable_accretion_efficiency' / s
                    else:   
                        print(f'Unknown subtype: {s}')
                        continue
                kw = {'metallicity': 0.02} if c == 'SEVN' else {}
                if c in ['SEVN']:
                    fname = path / f'{c}_MIST_T0.csv'    
                elif c in ['BPASS']:
                    fname = path / f'{c}_T0.csv'
                else:
                    fname = path / f'{c}_T0.hdf5'
                
                try:
                    T0, _ = load.load_T0_data(fname, c, **kw)
                except Exception as e:
                    print(f"Error loading T0 data for code {c} in run wave {rw} with subtype {s}: {e}")
                    continue
                
                print(f"T0 data loaded for code {c} in run wave {rw} with subtype {s}.")

                complicated_channel_ids = fc.first_interaction_channels(T0, verbose=False)
                print(f"Complicated channel IDs for code {c} in run wave {rw} with subtype {s} done")

                simple_channel_ids = fc.simplify_channels_dict(complicated_channel_ids)
                print(f"Simple channel IDs for code {c} in run wave {rw} with subtype {s} done")

                grp = hf.create_group(f'{c}_{s}')
                for key, arr in simple_channel_ids.items():
                    grp.create_dataset(key, data=np.array(arr))

T0 data loaded for code COSMIC in run wave initial_condition_variations with subtype fiducial.
Complicated channel IDs for code COSMIC in run wave initial_condition_variations with subtype fiducial done
Simple channel IDs for code COSMIC in run wave initial_condition_variations with subtype fiducial done
T0 data loaded for code BSE in run wave initial_condition_variations with subtype fiducial.
Complicated channel IDs for code BSE in run wave initial_condition_variations with subtype fiducial done
Simple channel IDs for code BSE in run wave initial_condition_variations with subtype fiducial done
T0 data loaded for code SEVN in run wave initial_condition_variations with subtype fiducial.
Complicated channel IDs for code SEVN in run wave initial_condition_variations with subtype fiducial done
Simple channel IDs for code SEVN in run wave initial_condition_variations with subtype fiducial done
T0 data loaded for code SeBa in run wave initial_condition_variations with subtype fiducial.
Comp

In [4]:
def load_channel_ids(filename, code, variation):
    """
    Load a single dictionary from an HDF5 file.

    Parameters
    ----------
    filename : str
        Path to the HDF5 file.
    code : str
        The formaton channel you want binary ids for
    variation : str
        The variation you want formaton channel ids for

    Returns
    -------
    dict
        Dictionary containing the stored data.
    """
    key = f'{code}_{variation}'
    with h5py.File(filename, "r") as f:
        if key not in f:
            raise KeyError(f"No group named '{key}' in {filename}")
        grp = f[key]
        return {k: grp[k][()] for k in grp}

In [5]:
for rw, subtype in zip(run_waves, [initial_conditions_variations, mass_transfer_variations]):
        for s in subtype:
            for c in codes:
                if rw == 'initial_condition_variations':
                    path = dat_path / rw / s
                elif rw == 'mass_transfer_variations':
                    if s in ['alpha_lambda_1', 'alpha_lambda_2', 'alpha_lambda_02', 'alpha_lambda_05', 'alpha_gamma_2']:
                        path = dat_path / rw / 'common_envelope' / s
                    elif s in ['qcrit_claeys_14', 'qcrit_hurley_02', 'qcrit_hurley_webbink', 'qcrit_zetas']:
                        path = dat_path / rw / 'stability_of_mass_transfer' / s
                    elif s in ['accretion_0', 'accretion_1', 'accretion_05']:
                        path = dat_path / rw / 'stable_accretion_efficiency' / s
                    else:   
                        print(f'Unknown subtype: {s}')
                        continue
                for key in ['CE_1', 'SMT_1', 'DCCE', 'nonRLO', 'mergers', 'other']:
                    try:
                        channel_ids = load_channel_ids(outfile, c, s)
                        print(f"Channel IDs for code {c} in run wave {rw} with subtype {s} loaded successfully.")
                    except Exception as e:
                        print(f"Error loading channel IDs for code {c} in run wave {rw} with subtype {s}: {e}")

Channel IDs for code COSMIC in run wave initial_condition_variations with subtype fiducial loaded successfully.
Channel IDs for code COSMIC in run wave initial_condition_variations with subtype fiducial loaded successfully.
Channel IDs for code COSMIC in run wave initial_condition_variations with subtype fiducial loaded successfully.
Channel IDs for code COSMIC in run wave initial_condition_variations with subtype fiducial loaded successfully.
Channel IDs for code COSMIC in run wave initial_condition_variations with subtype fiducial loaded successfully.
Channel IDs for code COSMIC in run wave initial_condition_variations with subtype fiducial loaded successfully.
Channel IDs for code BSE in run wave initial_condition_variations with subtype fiducial loaded successfully.
Channel IDs for code BSE in run wave initial_condition_variations with subtype fiducial loaded successfully.
Channel IDs for code BSE in run wave initial_condition_variations with subtype fiducial loaded successfully.
C

In [ ]:
def plot_dat_from_channel(data, channel, channel_ids, stage, xcolumn, ycolumn, log10_x=False, log10_y=False):
    ZAMS, WDMS, DWD = fc.select_evolutionary_states(d=data)
    data = []
    if stage == 'ZAMS':
      dat_select = ZAMS.loc[ZAMS.ID.isin(channel_ids[channel])].copy()
    elif stage == 'WDMS':
      dat_select = WDMS.loc[WDMS.ID.isin(channel_ids[channel])].copy()
    elif stage == 'DWD':
      dat_select = DWD.loc[DWD.ID.isin(channel_ids[channel])].copy()
    # add in mass ratio
    dat_select["q"] = np.minimum(dat_select["mass1"], dat_select["mass2"]) / np.maximum(dat_select["mass1"], dat_select["mass2"])
    if log10_x:
      plot_x = np.log10(dat_select[xcolumn])
    else:
      plot_x = dat_select[xcolumn]
    if log10_y:
      plot_y = np.log10(dat_select[ycolumn])
    else:
      plot_y = dat_select[ycolumn]
    if len(dat_select) > 100:
      plt.hist2d(plot_x, plot_y, bins=50, norm=colors.LogNorm())
      plt.colorbar(label='counts per bin')
    else:
      plt.scatter(plot_x, plot_y)
    plt.xlabel(xcolumn)
    plt.ylabel(ycolumn)
    plt.show()
  
    return

In [ ]:
c = codes[0]
f = fnames[0]

data_T0, _ = load.load_T0_data(dat_path+f, code=c)
complicated_channels = fc.first_interaction_channels(d=data_T0, verbose=False)
simple_channels = fc.simplify_channels_dict(channels_dict=complicated_channels)

In [ ]:
simple_channels

{'SMT_1': array([   108,    126,    194, ..., 629669, 629764, 629886]),
 'CE_1': array([     2,      4,     10, ..., 629865, 629876, 629881]),
 'DCCE': array([ 25103,  86064, 127202, 159964, 202723, 247641, 250997, 314520,
        357478, 554914, 555916, 619905]),
 'nonRLO': array([     0,      1,      5, ..., 629895, 629896, 629897]),
 'mergers': array([     3,     23,     47, ..., 595165, 610500, 618761]),
 'other': array([], dtype=int64)}

In [ ]:
plot = False
d_weird = []
for c, f in zip(codes, fnames):
    if c !='SEVN':
        data_T0, _ = load.load_T0_data(dat_path+f, code=c)
    else:
        data_T0, _ = load.load_T0_data(dat_path+f, code=c, metallicity=0.02)
    complicated_channels = fc.first_interaction_channels(d=data_T0, verbose=False)
    simple_channels = fc.simplify_channels_dict(channels_dict=complicated_channels)
    print(f'channels for code: {c}')
    for key, arr in simple_channels.items():
        print(key, len(arr))
    print()

    if len(simple_channels['other']) > 0:
        d_weird.append(data_T0.loc[data_T0.ID.isin(simple_channels['other'])])
        
#    if plot:
#        for key, arr in simple_channels.items():
#            print(f'plotting channel: {key} for {c}')
#            _ = plot_dat_from_channel(
#                data = data_T0, channel=key, channel_ids=simple_channels, stage='ZAMS',
#                xcolumn='q', ycolumn='semiMajor', log10_x=False, log10_y=True)

channels for code: COSMIC
SMT_1 7712
CE_1 69337
DCCE 12
nonRLO 525981
mergers 26856
other 0

channels for code: BSE
SMT_1 53213
CE_1 29186
DCCE 25
nonRLO 524501
mergers 22973
other 0

channels for code: SEVN
SMT_1 8653
CE_1 38166
DCCE 0
nonRLO 217003
mergers 15053
other 39

channels for code: BPASS
SMT_1 2907
CE_1 976
DCCE 0
nonRLO 6664
mergers 0
other 18

channels for code: SeBa
SMT_1 6348
CE_1 52791
DCCE 1951
nonRLO 541781
mergers 27025
other 2

channels for code: COMPAS
SMT_1 23167
CE_1 52567
DCCE 4
nonRLO 542166
mergers 11876
other 120

channels for code: METISSE
SMT_1 2666
CE_1 38237
DCCE 24
nonRLO 565014
mergers 23956
other 1



In [ ]:
d_sevn = d_weird[0]
d_sevn_DWD_id = d_sevn.loc[(d_sevn.type1.isin([21, 22, 23])) & (d_sevn.type2.isin([21,22,2])) & (d_sevn.semiMajor>0)].groupby('ID', as_index=False).first().ID
for ID in d_sevn_DWD_id[::5]:
    print(d_sevn.loc[d_sevn.ID == ID][['time', 'type1', 'type2', 'event', 'mass1', 'mass2', 'semiMajor']])

            time  type1  type2  event     mass1     mass2   semiMajor
694236     0.000    121    121     13  1.872583  1.806317  269.392200
694237  1303.532    121    121     10  1.871662  1.805573  269.513800
694238  1305.836    122    121     10  1.871658  1.805571  269.514300
694239  1403.355    124    121     10  1.861380  1.806663  266.580700
694240  1442.680    124    121     10  1.860922  1.806625  266.675100
694241  1445.518    124    122     10  1.860889  1.806622  266.678200
694242  1552.264    124    123     10  1.859542  1.804860  266.065500
694243  1552.382   1251    123     10  1.859537  1.804847  266.056800
694244  1566.095   1251    123     32  1.859000  1.796000  256.186000
694245  1566.095   1251     21    511  0.506368  0.424418    2.538718
694246  1566.095    132     21     42  0.506400  0.424400    2.538720
694247  1568.163     22     21     10  0.505497  0.424418    2.541093
694248  1568.163     22     21     82  0.505497  0.424418    2.541093
             time  t

It looks like the event logging might be missing some Roche overflows from AGBs? I'm surprised that the orbit is shrinking as the TPAGB removes the envelope. 

In [ ]:
d = d_weird[1]
d_DWD_id = d.loc[(d.type1.isin([21, 22, 23])) & (d.type2.isin([21,22,2])) & (d.semiMajor>0)].groupby('ID', as_index=False).first().ID
for ID in d_DWD_id[::2]:
    print(d.loc[d.ID == ID][['time', 'type1', 'type2', 'event', 'mass1', 'mass2', 'semiMajor']])

          time  type1  type2  event     mass1     mass2  semiMajor
15400     0.00     12     12     13  1.261000  1.217000    502.516
15401  6624.97     21     12     11  0.517559  1.217000    509.365
15402  6625.44     21     12     11  0.874891  1.217000    509.365
15403  7635.57     21     12     32  0.874891  0.701963    675.695
15404  7635.57     21     12     42  0.874907  0.566587    716.908
15405  7635.57     21     22     11  0.874907  0.544965    727.303
          time  type1  type2  event     mass1     mass2  semiMajor
15887     0.00     12     12     13  1.277000  1.194000    501.518
15888  6344.02     21     12     11  0.517524  1.194000    508.254
15889  6348.66     21     12     11  0.881046  1.194000    508.254
15890  8174.56     21     12     32  0.881046  0.696175    668.633
15891  8174.56     21     12     42  0.881062  0.564482    707.347
15892  8174.56     21     22     11  0.881062  0.543762    717.096
          time  type1  type2  event     mass1     mass2  semiM

I think something werid is hapening with the primaries here; they are going down to 0.5 Msun, then back up to ~1.05 Msun. 

In [ ]:
d_seba = d_weird[2]
print(d_seba[['time', 'type1', 'type2', 'event', 'mass1', 'mass2', 'semiMajor']])
#d_seba_DWD_id = d_seba.loc[(d_seba.type1.isin([21, 22, 23])) & (d_seba.type2.isin([21,22,2])) & (d_seba.semiMajor>0)].groupby('ID', as_index=False).first().ID
#for ID in d_seba_DWD_id:
#    print(d_seba.loc[d_seba.ID == ID][['time', 'type1', 'type2', 'event', 'mass1', 'mass2', 'semiMajor']])

               time  type1  type2  event    mass1    mass2   semiMajor
2217937      0.0000  121.0  121.0   13.0  7.84165  7.59547   1644.4200
2217938     38.6963  122.0  121.0   11.0  7.82358  7.58496   1647.4700
2217939     38.8138  123.0  121.0   11.0  7.82250  7.58483   1647.6000
2217940     38.8447  124.0  121.0   11.0  7.82134  7.58480   1647.7300
2217941     41.2820  124.0  122.0   12.0  7.75252  7.58175   1655.4500
2217942     41.4099  124.0  123.0   12.0  7.75065  7.58088   1655.7500
2217943     41.4458  124.0  124.0   12.0  7.75013  7.57969   1655.9300
2217944     43.4780  125.0  124.0   11.0  7.69560  7.52188   1668.1600
2217945     43.7031    3.0  124.0   11.0  1.18897  7.51761  10639.2000
2217946     46.4288    3.0  125.0   12.0  1.18897  7.45922   1330.1400
2217947     46.8905    3.0  125.0   32.0  1.18897  7.25494   1319.2000
2217948     46.8905    3.0  125.0  512.0  1.18897  7.25494   1319.2000
2217949     46.8905    3.0   23.0   12.0  1.18897  1.35396     24.0664
221795

These make NS binaries so I'm not worried about it. 

In [ ]:
d = d_weird[3]
d_DWD_id = d.loc[(d.type1.isin([21, 22, 23])) & (d.type2.isin([21,22,2])) & (d.semiMajor>0)].groupby('ID', as_index=False).first().ID
for ID in d_DWD_id[::10]:
    print(d.loc[d.ID == ID][['time', 'type1', 'type2', 'event', 'mass1', 'mass2', 'semiMajor']])

             time   type1   type2  event     mass1     mass2    semiMajor
9227     0.000000   121.0   121.0   13.0  2.092033  1.944561   640.096080
9228   966.502495   122.0   121.0   11.0  2.091350  1.944178   640.265685
9229   974.287406   123.0   121.0   11.0  2.091214  1.944178   640.287203
9230   989.507233   124.0   121.0   11.0  2.089543  1.944178   640.552489
9231  1190.022288   124.0   122.0   12.0  2.077363  1.944178   642.492537
9232  1200.157392   124.0   123.0   12.0  2.076204  1.944047   642.698825
9233  1225.957423   124.0   124.0   12.0  2.072296  1.941399   643.748503
9234  1225.957423   124.0  1251.0   12.0  2.072200  1.941378   643.748503
9235  1230.802354   124.0  1252.0   12.0  2.071350  1.912870   648.511028
9236  1231.689854   124.0    22.0   12.0  2.071167  0.584964   668.097207
9237  1231.689854   124.0    22.0   32.0  2.084996  0.584964  2832.521212
9238  1231.689854   124.0    22.0   41.0  2.084996  0.584964  2832.521212
9239  1247.260364  1251.0    22.0   11

This looks like the dominant factor is that the secondary is evolving faaster than the primary; not sure why this is happening. 

In [ ]:
d = d_weird[4]
print(d[['time', 'type1', 'type2', 'event', 'mass1', 'mass2', 'semiMajor']])


                time   type1   type2  event     mass1     mass2     semiMajor
268138      0.000000   121.0   121.0   13.0  2.033649  1.894859    260.675156
268138   1006.405440   122.0   121.0   12.0  2.032648  1.894195    260.785665
268138   1020.262636   123.0   121.0   12.0  2.032605  1.894179    260.789461
268138   1040.664840   124.0   121.0   12.0  2.032169  1.894168    262.659556
268138   1218.827256   124.0   122.0   12.0  2.030770  1.893945    263.893105
268138   1239.201752   124.0   123.0   12.0  2.030621  1.893901    263.910301
268138   1304.255869   124.0   124.0   12.0  2.031271  1.884840    257.523113
268138   1305.438736   124.0   124.0   32.0  2.031431  1.884075    256.382828
268138   1308.417357   124.0   124.0   42.0  2.035907  1.879525    256.213275
268138   1353.672110  1251.0   124.0   12.0  2.035502  1.878903    266.763130
268138   1373.602030    22.0   124.0   12.0  0.392292  1.878936    260.316230
268138   1457.630272    22.0  1251.0   12.0  0.392292  1.877855 